In [1]:
# Phase A:
#   Original frozen 12-case cohort
#   3 predictive models
#   lambda = 0.0 and 1.0
#
# Purpose:
#   Extend realised validation beyond the existing lambda=0.5
#   comparison, while preserving the final production MILP
#   formulation and computational budget.
#
# IMPORTANT:
#   - Does NOT modify the frozen production experiment.
#   - Does NOT use actual durations during optimisation.
#   - Actual durations are introduced ONLY after the planned
#     schedule has been generated.
# ============================================================

import os
import time
import numpy as np
import pandas as pd

# Load final frozen MILP engine.
%run milp_engine.ipynb

print("Final MILP engine loaded.")

Note: you may need to restart the kernel to use updated packages.
Final MILP engine loaded.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# ============================================================
# 1. FROZEN VALIDATION CONFIGURATION
# ============================================================

MODELS = {
    "LightGBM":
        "Prescriptive/LightGBM_prescriptive_optimizer_inputs.csv",

    "RF":
        "Prescriptive/RF_prescriptive_optimizer_inputs.csv",

    "XGBoost":
        "Prescriptive/XGBoost_prescriptive_optimizer_inputs.csv"
}


# ------------------------------------------------------------
# Phase A endpoint risk levels
#
# lambda = 0.0 -> P50 planning
# lambda = 1.0 -> P90 planning
#
# Existing lambda = 0.5 realised results are NOT rerun here.
# ------------------------------------------------------------

VALIDATION_LAMBDAS = [
    0.0,
    1.0
]


# ------------------------------------------------------------
# Final MILP configuration
# ------------------------------------------------------------

N_CASES = 12

N_ROOMS = 3

ROOM_CAPACITY = 480

TURNOVER = 20

BALANCE_WEIGHT = 0.10

DURATION_CAP = 360.0

TIME_LIMIT = 300


# ------------------------------------------------------------
# Input files
# ------------------------------------------------------------

COHORT_PATH = (
    "Cohort/fixed_cohort.csv"
)

ACTUAL_TEST_PATH = (
    "new_data/mover_epic_final_test_features.csv"
)


# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

RESULTS_DIR = "Results"

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)


SCHEDULE_OUTPUT = os.path.join(
    RESULTS_DIR,
    "additional_realised_validation_schedules.csv"
)

ROOM_OUTPUT = os.path.join(
    RESULTS_DIR,
    "additional_realised_validation_room_results.csv"
)

SUMMARY_OUTPUT = os.path.join(
    RESULTS_DIR,
    "additional_realised_validation_summary.csv"
)

CHECKPOINT_OUTPUT = os.path.join(
    RESULTS_DIR,
    "additional_realised_validation_checkpoint.csv"
)


print("=" * 75)
print("STEP 4C — REALISED VALIDATION CONFIGURATION")
print("=" * 75)

print(f"Models                : {list(MODELS.keys())}")
print(f"Lambda values         : {VALIDATION_LAMBDAS}")
print(f"Cases                 : {N_CASES}")
print(f"Rooms                 : {N_ROOMS}")
print(f"Room capacity         : {ROOM_CAPACITY} min")
print(f"Turnover              : {TURNOVER} min")
print(f"Balance weight beta   : {BALANCE_WEIGHT}")
print(f"Duration cap          : {DURATION_CAP} min")
print(f"Solver time limit     : {TIME_LIMIT} s")
print(
    f"Planned new MILP runs: "
    f"{len(MODELS) * len(VALIDATION_LAMBDAS)}"
)

STEP 4C — REALISED VALIDATION CONFIGURATION
Models                : ['LightGBM', 'RF', 'XGBoost']
Lambda values         : [0.0, 1.0]
Cases                 : 12
Rooms                 : 3
Room capacity         : 480 min
Turnover              : 20 min
Balance weight beta   : 0.1
Duration cap          : 360.0 min
Solver time limit     : 300 s
Planned new MILP runs: 6


In [4]:
# ============================================================
# 2. LOAD AND PROTECT ORIGINAL FROZEN COHORT
# ============================================================

assert os.path.exists(COHORT_PATH), (
    f"Frozen cohort not found: {COHORT_PATH}"
)

cohort_df = pd.read_csv(
    COHORT_PATH
)


assert "LOG_ID" in cohort_df.columns, (
    "fixed_cohort.csv must contain LOG_ID."
)

assert len(cohort_df) == N_CASES, (
    f"Expected {N_CASES} cases, "
    f"found {len(cohort_df)}."
)

assert cohort_df["LOG_ID"].notna().all(), (
    "Missing LOG_ID in frozen cohort."
)

assert cohort_df["LOG_ID"].is_unique, (
    "Duplicate LOG_ID in frozen cohort."
)


# Standardise IDs as strings.

frozen_ids = (
    cohort_df["LOG_ID"]
    .astype(str)
    .tolist()
)


print("=" * 75)
print("FROZEN COHORT CHECK: PASS")
print("=" * 75)

print(f"Cohort size: {len(frozen_ids)}")

print("\nFrozen LOG_ID order:")

for idx, log_id in enumerate(
    frozen_ids,
    start=1
):
    print(
        f"{idx:02d}: {log_id}"
    )

FROZEN COHORT CHECK: PASS
Cohort size: 12

Frozen LOG_ID order:
01: f42047cb579b3055
02: 28a358995e41a9f5
03: b29ebd9769926496
04: de7a28cae633f05a
05: 564d6a7a6173eed1
06: a8da758576964b78
07: 5a8e1df80d6b047f
08: ec5cecc91ac56559
09: cf2dd7eb2a698665
10: 4cc0c8e2dd8ffc9c
11: d6b5106aa593a6fc
12: fc7299c4e0060a00


In [5]:
# ============================================================
# 3. LOAD ACTUAL DURATIONS
#
# IMPORTANT:
# ACTUAL_DURATION is used ONLY for post-optimisation replay.
# It must never enter solve_or_milp().
# ============================================================

assert os.path.exists(ACTUAL_TEST_PATH), (
    f"Final test cohort not found: {ACTUAL_TEST_PATH}"
)

actual_df = pd.read_csv(
    ACTUAL_TEST_PATH
)


required_actual_columns = {
    "LOG_ID",
    "ACTUAL_DURATION"
}

missing_actual_columns = (
    required_actual_columns
    -
    set(actual_df.columns)
)

assert not missing_actual_columns, (
    f"Missing columns in final test cohort: "
    f"{missing_actual_columns}"
)


# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

assert len(actual_df) == 17083, (
    f"Expected final held-out cohort size 17083, "
    f"found {len(actual_df)}."
)

assert actual_df["LOG_ID"].notna().all()

assert actual_df["LOG_ID"].is_unique

assert actual_df["ACTUAL_DURATION"].notna().all()

assert np.isfinite(
    actual_df["ACTUAL_DURATION"]
    .to_numpy(dtype=float)
).all()

assert (
    actual_df["ACTUAL_DURATION"] > 0
).all()


actual_df = actual_df.copy()

actual_df["LOG_ID"] = (
    actual_df["LOG_ID"]
    .astype(str)
)


# ------------------------------------------------------------
# Extract original frozen cohort
# ------------------------------------------------------------

missing_actual_ids = (
    set(frozen_ids)
    -
    set(actual_df["LOG_ID"])
)

assert not missing_actual_ids, (
    "Frozen cohort IDs missing from final test cohort:\n"
    f"{sorted(missing_actual_ids)}"
)


actual_fixed = (
    actual_df
    .set_index("LOG_ID")
    .loc[frozen_ids]
    .reset_index()
)


assert len(actual_fixed) == N_CASES

assert (
    actual_fixed["LOG_ID"].tolist()
    ==
    frozen_ids
)


actual_duration_map = dict(
    zip(
        actual_fixed["LOG_ID"],
        actual_fixed["ACTUAL_DURATION"]
        .astype(float)
    )
)


print("=" * 75)
print("ACTUAL-DURATION AUDIT: PASS")
print("=" * 75)

print(
    f"Final held-out cohort: {len(actual_df)}"
)

print(
    f"Frozen validation cases: {len(actual_fixed)}"
)

print(
    f"Total actual surgical duration: "
    f"{actual_fixed['ACTUAL_DURATION'].sum():.2f} min"
)

print(
    f"Nominal three-room capacity: "
    f"{N_ROOMS * ROOM_CAPACITY:.2f} min"
)

print(
    f"Actual workload / nominal capacity: "
    f"{actual_fixed['ACTUAL_DURATION'].sum() / (N_ROOMS * ROOM_CAPACITY):.4f}"
)

ACTUAL-DURATION AUDIT: PASS
Final held-out cohort: 17083
Frozen validation cases: 12
Total actual surgical duration: 2192.00 min
Nominal three-room capacity: 1440.00 min
Actual workload / nominal capacity: 1.5222


In [6]:
# ============================================================
# 4. VERIFY FINAL OPTIMIZER INPUTS
# ============================================================

verified_model_data = {}


for model_name, model_path in MODELS.items():

    print("\n" + "-" * 75)
    print(f"Verifying: {model_name}")
    print("-" * 75)

    assert os.path.exists(model_path), (
        f"Missing optimizer input: {model_path}"
    )

    df = pd.read_csv(
        model_path
    )


    required_columns = {
        "LOG_ID",
        "DURATION_P50_MINS",
        "DURATION_P90_MINS"
    }

    missing_columns = (
        required_columns
        -
        set(df.columns)
    )

    assert not missing_columns, (
        f"{model_name}: missing columns "
        f"{missing_columns}"
    )


    # --------------------------------------------------------
    # ID integrity
    # --------------------------------------------------------

    assert df["LOG_ID"].notna().all()

    assert df["LOG_ID"].is_unique


    df = df.copy()

    df["LOG_ID"] = (
        df["LOG_ID"]
        .astype(str)
    )


    # --------------------------------------------------------
    # Prediction integrity
    # --------------------------------------------------------

    pred_cols = [
        "DURATION_P50_MINS",
        "DURATION_P90_MINS"
    ]

    assert df[pred_cols].notna().all().all()

    assert np.isfinite(
        df[pred_cols]
        .to_numpy(dtype=float)
    ).all()

    assert (
        df["DURATION_P50_MINS"] > 0
    ).all()

    assert (
        df["DURATION_P90_MINS"]
        >=
        df["DURATION_P50_MINS"]
    ).all()


    # --------------------------------------------------------
    # Frozen cohort availability
    # --------------------------------------------------------

    missing_ids = (
        set(frozen_ids)
        -
        set(df["LOG_ID"])
    )

    assert not missing_ids, (
        f"{model_name}: missing frozen IDs:\n"
        f"{sorted(missing_ids)}"
    )


    # --------------------------------------------------------
    # Extract EXACT same 12 cases in EXACT same order
    # --------------------------------------------------------

    df_day = (
        df
        .set_index("LOG_ID")
        .loc[frozen_ids]
        .reset_index()
    )


    assert len(df_day) == N_CASES

    assert (
        df_day["LOG_ID"].tolist()
        ==
        frozen_ids
    )


    verified_model_data[
        model_name
    ] = df_day


    print(
        f"{model_name}: PASS "
        f"| full={df.shape} "
        f"| fixed={df_day.shape}"
    )


# ------------------------------------------------------------
# Cross-model alignment
# ------------------------------------------------------------

reference_ids = (
    verified_model_data["LightGBM"]
    ["LOG_ID"]
    .tolist()
)


for model_name in MODELS:

    assert (
        verified_model_data[
            model_name
        ]["LOG_ID"].tolist()
        ==
        reference_ids
    )


print("\n" + "=" * 75)
print("CROSS-MODEL INPUT AUDIT: PASS")
print("=" * 75)


---------------------------------------------------------------------------
Verifying: LightGBM
---------------------------------------------------------------------------
LightGBM: PASS | full=(17083, 4) | fixed=(12, 4)

---------------------------------------------------------------------------
Verifying: RF
---------------------------------------------------------------------------
RF: PASS | full=(17083, 4) | fixed=(12, 4)

---------------------------------------------------------------------------
Verifying: XGBoost
---------------------------------------------------------------------------
XGBoost: PASS | full=(17083, 4) | fixed=(12, 4)

CROSS-MODEL INPUT AUDIT: PASS


In [7]:
# ============================================================
# 5. REALISED SCHEDULE REPLAY FUNCTION
#
# Planned room assignment and planned within-room order are
# frozen from the MILP solution.
#
# Actual surgical durations are then introduced ex post.
#
# A case cannot start:
#   1. before its planned start time; or
#   2. before the previous realised case has completed plus
#      the required turnover.
#
# Therefore:
#
# realised_start_i =
#     max(
#         planned_start_i,
#         previous_realised_end + turnover
#     )
#
# No re-optimisation is performed using actual durations.
# ============================================================


def replay_realised_schedule(
    df_day,
    milp_result,
    actual_duration_map,
    model_name,
    lam,
    cohort_name="Original_Fixed_Cohort",
    n_rooms=3,
    room_capacity=480,
    turnover=20,
    duration_cap=360.0
):

    # --------------------------------------------------------
    # Extract MILP schedule
    # --------------------------------------------------------

    room_assignment = (
        milp_result["room_assignment"]
    )

    start_times = (
        milp_result["start_times"]
    )


    assert len(room_assignment) == len(df_day)

    assert len(start_times) == len(df_day)


    # --------------------------------------------------------
    # Reconstruct planning durations exactly as used by MILP
    # --------------------------------------------------------

    p50 = (
        df_day["DURATION_P50_MINS"]
        .to_numpy(dtype=float)
    )

    p90 = (
        df_day["DURATION_P90_MINS"]
        .to_numpy(dtype=float)
    )

    planning_uncapped = (
        p50
        +
        float(lam)
        *
        (p90 - p50)
    )


    if duration_cap is None:

        planning_duration = (
            planning_uncapped.copy()
        )

        capped_mask = np.zeros(
            len(df_day),
            dtype=bool
        )

    else:

        capped_mask = (
            planning_uncapped
            >
            float(duration_cap)
        )

        planning_duration = np.minimum(
            planning_uncapped,
            float(duration_cap)
        )


    # --------------------------------------------------------
    # Build planned schedule table
    # --------------------------------------------------------

    schedule_rows = []


    for i in range(len(df_day)):

        log_id = str(
            df_day.iloc[i]["LOG_ID"]
        )

        assert log_id in actual_duration_map, (
            f"Missing actual duration for {log_id}"
        )


        schedule_rows.append({

            "Cohort":
                cohort_name,

            "Model":
                model_name,

            "Lambda":
                float(lam),

            "Case_Index":
                int(i),

            "LOG_ID":
                log_id,

            "Room":
                int(room_assignment[i]),

            "Planned_Start":
                float(start_times[i]),

            "P50_Duration":
                float(p50[i]),

            "P90_Duration":
                float(p90[i]),

            "Planning_Duration_Uncapped":
                float(planning_uncapped[i]),

            "Planning_Duration":
                float(planning_duration[i]),

            "Duration_Was_Capped":
                bool(capped_mask[i]),

            "Actual_Duration":
                float(
                    actual_duration_map[
                        log_id
                    ]
                )
        })


    schedule_df = pd.DataFrame(
        schedule_rows
    )


    # --------------------------------------------------------
    # Planned order:
    #
    # Sort by MILP planned start time within each room.
    #
    # This is sufficient because final MILP sequencing
    # constraints ensure non-overlapping room schedules.
    # --------------------------------------------------------

    schedule_df = (
        schedule_df
        .sort_values(
            [
                "Room",
                "Planned_Start",
                "Case_Index"
            ]
        )
        .reset_index(drop=True)
    )


    schedule_df[
        "Sequence_Position"
    ] = (
        schedule_df
        .groupby("Room")
        .cumcount()
        + 1
    )


    # --------------------------------------------------------
    # Realised replay
    # --------------------------------------------------------

    schedule_df[
        "Realised_Start"
    ] = np.nan

    schedule_df[
        "Realised_End"
    ] = np.nan

    schedule_df[
        "Start_Delay"
    ] = np.nan


    for room in range(n_rooms):

        room_indices = (
            schedule_df.index[
                schedule_df["Room"]
                ==
                room
            ]
            .tolist()
        )


        previous_realised_end = None


        for row_idx in room_indices:

            planned_start = float(
                schedule_df.at[
                    row_idx,
                    "Planned_Start"
                ]
            )

            actual_duration = float(
                schedule_df.at[
                    row_idx,
                    "Actual_Duration"
                ]
            )


            if previous_realised_end is None:

                realised_start = (
                    planned_start
                )

            else:

                realised_start = max(
                    planned_start,
                    previous_realised_end
                    +
                    turnover
                )


            realised_end = (
                realised_start
                +
                actual_duration
            )


            start_delay = (
                realised_start
                -
                planned_start
            )


            schedule_df.at[
                row_idx,
                "Realised_Start"
            ] = realised_start

            schedule_df.at[
                row_idx,
                "Realised_End"
            ] = realised_end

            schedule_df.at[
                row_idx,
                "Start_Delay"
            ] = start_delay


            previous_realised_end = (
                realised_end
            )


    # --------------------------------------------------------
    # Defensive replay checks
    # --------------------------------------------------------

    assert (
        schedule_df["Realised_Start"]
        >=
        schedule_df["Planned_Start"]
        -
        1e-7
    ).all(), (
        "Realised start before planned start detected."
    )


    assert (
        schedule_df["Start_Delay"]
        >=
        -1e-7
    ).all(), (
        "Negative realised start delay detected."
    )


    # --------------------------------------------------------
    # Explicit overlap / turnover audit
    # --------------------------------------------------------

    turnover_violations = []


    for room in range(n_rooms):

        room_df = (
            schedule_df[
                schedule_df["Room"]
                ==
                room
            ]
            .sort_values(
                "Sequence_Position"
            )
            .reset_index(drop=True)
        )


        for j in range(
            1,
            len(room_df)
        ):

            previous_end = float(
                room_df.loc[
                    j - 1,
                    "Realised_End"
                ]
            )

            current_start = float(
                room_df.loc[
                    j,
                    "Realised_Start"
                ]
            )


            required_start = (
                previous_end
                +
                turnover
            )


            if (
                current_start
                <
                required_start
                -
                1e-6
            ):

                turnover_violations.append({

                    "Room":
                        room,

                    "Previous_LOG_ID":
                        room_df.loc[
                            j - 1,
                            "LOG_ID"
                        ],

                    "Current_LOG_ID":
                        room_df.loc[
                            j,
                            "LOG_ID"
                        ],

                    "Required_Start":
                        required_start,

                    "Actual_Start":
                        current_start
                })


    assert len(turnover_violations) == 0, (
        "Realised overlap / turnover violation detected:\n"
        f"{turnover_violations}"
    )


    # --------------------------------------------------------
    # Room-level realised metrics
    # --------------------------------------------------------

    room_rows = []


    for room in range(n_rooms):

        room_df = (
            schedule_df[
                schedule_df["Room"]
                ==
                room
            ]
            .sort_values(
                "Sequence_Position"
            )
        )


        if len(room_df) == 0:

            realised_finish = 0.0

            actual_workload = 0.0

            total_delay = 0.0

            max_delay = 0.0

            mean_delay = 0.0

            n_cases_room = 0

        else:

            realised_finish = float(
                room_df[
                    "Realised_End"
                ]
                .max()
            )

            actual_workload = float(
                room_df[
                    "Actual_Duration"
                ]
                .sum()
            )

            total_delay = float(
                room_df[
                    "Start_Delay"
                ]
                .sum()
            )

            max_delay = float(
                room_df[
                    "Start_Delay"
                ]
                .max()
            )

            mean_delay = float(
                room_df[
                    "Start_Delay"
                ]
                .mean()
            )

            n_cases_room = int(
                len(room_df)
            )


        realised_overtime = max(
            realised_finish
            -
            room_capacity,
            0.0
        )


        room_rows.append({

            "Cohort":
                cohort_name,

            "Model":
                model_name,

            "Lambda":
                float(lam),

            "Room":
                int(room),

            "N_Cases":
                n_cases_room,

            "Actual_Surgical_Workload":
                actual_workload,

            "Realised_Room_Finish":
                realised_finish,

            "Realised_Overtime":
                realised_overtime,

            "Total_Start_Delay":
                total_delay,

            "Mean_Start_Delay":
                mean_delay,

            "Max_Start_Delay":
                max_delay
        })


    room_df = pd.DataFrame(
        room_rows
    )


    # --------------------------------------------------------
    # Run-level realised metrics
    # --------------------------------------------------------

    realised_makespan = float(
        room_df[
            "Realised_Room_Finish"
        ]
        .max()
    )


    realised_total_overtime = float(
        room_df[
            "Realised_Overtime"
        ]
        .sum()
    )


    total_start_delay = float(
        schedule_df[
            "Start_Delay"
        ]
        .sum()
    )


    mean_start_delay = float(
        schedule_df[
            "Start_Delay"
        ]
        .mean()
    )


    max_start_delay = float(
        schedule_df[
            "Start_Delay"
        ]
        .max()
    )


    actual_total_duration = float(
        schedule_df[
            "Actual_Duration"
        ]
        .sum()
    )


    nominal_capacity = float(
        n_rooms
        *
        room_capacity
    )


    actual_capacity_load_ratio = (
        actual_total_duration
        /
        nominal_capacity
    )


    delayed_cases = int(
        (
            schedule_df[
                "Start_Delay"
            ]
            >
            1e-6
        )
        .sum()
    )


    # --------------------------------------------------------
    # IMPORTANT:
    #
    # This is retained only for comparability with the earlier
    # realised-validation notebook.
    #
    # It is NOT the original MILP objective.
    # Primary interpretation should use realised overtime,
    # makespan and delay directly.
    # --------------------------------------------------------

    realised_evaluation_score = (
        realised_total_overtime
        +
        0.5
        *
        realised_makespan
    )


    summary = {

        "Cohort":
            cohort_name,

        "Model":
            model_name,

        "Lambda":
            float(lam),

        "N_Cases":
            int(len(schedule_df)),

        "Actual_Total_Duration":
            actual_total_duration,

        "Nominal_Total_Capacity":
            nominal_capacity,

        "Actual_Capacity_Load_Ratio":
            actual_capacity_load_ratio,

        "Realised_Makespan":
            realised_makespan,

        "Realised_Overtime":
            realised_total_overtime,

        "Total_Start_Delay":
            total_start_delay,

        "Mean_Start_Delay":
            mean_start_delay,

        "Max_Start_Delay":
            max_start_delay,

        "Delayed_Cases":
            delayed_cases,

        "Delayed_Case_Rate":
            delayed_cases
            /
            len(schedule_df),

        "Realised_Evaluation_Score":
            realised_evaluation_score,

        "Replay_Turnover_Violations":
            len(turnover_violations),

        # MILP diagnostics
        "MILP_Objective":
            float(
                milp_result[
                    "Objective"
                ]
            ),

        "MILP_Planned_Overtime":
            float(
                milp_result[
                    "Overtime"
                ]
            ),

        "MILP_Planned_Makespan":
            float(
                milp_result[
                    "Makespan"
                ]
            ),

        "MILP_Gap":
            float(
                milp_result[
                    "Gap"
                ]
            ),

        "MILP_Solve_Time":
            float(
                milp_result[
                    "Solve_Time"
                ]
            ),

        "MILP_Status":
            int(
                milp_result[
                    "Status"
                ]
            ),

        "MILP_Hit_Time_Limit":
            bool(
                milp_result[
                    "Hit_Time_Limit"
                ]
            ),

        "MILP_Reached_1pct_Gap":
            bool(
                milp_result[
                    "Reached_1pct_Gap"
                ]
            ),

        "MILP_N_Capped":
            int(
                milp_result[
                    "N_Capped"
                ]
            ),

        "MILP_Pct_Capped":
            float(
                milp_result[
                    "Pct_Capped"
                ]
            ),

        "MILP_Max_Uncapped_Duration":
            float(
                milp_result[
                    "Max_Uncapped_Duration"
                ]
            ),

        "MILP_Max_Planning_Duration":
            float(
                milp_result[
                    "Max_Planning_Duration"
                ]
            )
    }


    return (
        schedule_df,
        room_df,
        summary
    )

In [8]:
# ============================================================
# 6. SMOKE TEST
#
# Run ONE instance only:
# LightGBM | Original cohort | lambda = 0.0
#
# Expected runtime: approximately 300 seconds.
# ============================================================

SMOKE_MODEL = "LightGBM"

SMOKE_LAMBDA = 0.0


smoke_df_day = (
    verified_model_data[
        SMOKE_MODEL
    ]
    .copy()
)


print("=" * 75)
print("STARTING STEP 4C SMOKE TEST")
print("=" * 75)

print(
    f"Model  : {SMOKE_MODEL}"
)

print(
    f"Lambda : {SMOKE_LAMBDA}"
)

print(
    f"Cases  : {len(smoke_df_day)}"
)


smoke_start = time.time()


smoke_result = solve_or_milp(

    df_day=smoke_df_day,

    lam=SMOKE_LAMBDA,

    n_rooms=N_ROOMS,

    room_capacity=ROOM_CAPACITY,

    turnover=TURNOVER,

    balance_weight=BALANCE_WEIGHT,

    duration_cap=DURATION_CAP,

    time_limit=TIME_LIMIT
)


smoke_elapsed = (
    time.time()
    -
    smoke_start
)


assert smoke_result is not None, (
    "Smoke-test MILP returned no feasible incumbent."
)


print(
    f"\nMILP finished in "
    f"{smoke_elapsed:.2f} seconds."
)


print(
    f"Solver gap: "
    f"{smoke_result['Gap']:.4f}"
)


print(
    f"Hit time limit: "
    f"{smoke_result['Hit_Time_Limit']}"
)

STARTING STEP 4C SMOKE TEST
Model  : LightGBM
Lambda : 0.0
Cases  : 12
Set parameter Username
Set parameter LicenseID to value 2835369
Academic license - for non-commercial use only - expires 2027-06-15

MILP finished in 301.44 seconds.
Solver gap: 0.6004
Hit time limit: True


In [9]:
# ============================================================
# 7. REPLAY SMOKE-TEST SCHEDULE USING ACTUAL DURATIONS
# ============================================================

(
    smoke_schedule,
    smoke_rooms,
    smoke_summary
) = replay_realised_schedule(

    df_day=smoke_df_day,

    milp_result=smoke_result,

    actual_duration_map=actual_duration_map,

    model_name=SMOKE_MODEL,

    lam=SMOKE_LAMBDA,

    cohort_name="Original_Fixed_Cohort",

    n_rooms=N_ROOMS,

    room_capacity=ROOM_CAPACITY,

    turnover=TURNOVER,

    duration_cap=DURATION_CAP
)


print("\n" + "=" * 75)
print("SMOKE TEST REALISED VALIDATION")
print("=" * 75)


for key, value in smoke_summary.items():

    print(
        f"{key:32s}: {value}"
    )


print("\n" + "=" * 75)
print("CASE-LEVEL REALISED SCHEDULE")
print("=" * 75)


display(
    smoke_schedule[
        [
            "LOG_ID",
            "Room",
            "Sequence_Position",
            "Planned_Start",
            "Planning_Duration",
            "Actual_Duration",
            "Realised_Start",
            "Realised_End",
            "Start_Delay"
        ]
    ]
)


print("\n" + "=" * 75)
print("ROOM-LEVEL REALISED RESULTS")
print("=" * 75)


display(
    smoke_rooms
)


SMOKE TEST REALISED VALIDATION
Cohort                          : Original_Fixed_Cohort
Model                           : LightGBM
Lambda                          : 0.0
N_Cases                         : 12
Actual_Total_Duration           : 2192.0
Nominal_Total_Capacity          : 1440.0
Actual_Capacity_Load_Ratio      : 1.5222222222222221
Realised_Makespan               : 1125.0
Realised_Overtime               : 1250.0526598634356
Total_Start_Delay               : 718.5140942355218
Mean_Start_Delay                : 59.87617451962682
Max_Start_Delay                 : 392.13936932467243
Delayed_Cases                   : 4
Delayed_Case_Rate               : 0.3333333333333333
Realised_Evaluation_Score       : 1812.5526598634356
Replay_Turnover_Violations      : 0
MILP_Objective                  : 1280.2211414717526
MILP_Planned_Overtime           : 891.5709721690033
MILP_Planned_Makespan           : 777.2645239338589
MILP_Gap                        : 0.6003971710486974
MILP_Solve_Time     

,LOG_ID,Room,Sequence_Position,Planned_Start,Planning_Duration,Actual_Duration,Realised_Start,Realised_End,Start_Delay
0,d6b5106aa593a6fc,0,1,0.000000,165.707943,251.0,0.000000,251.000000,0.000000
1,b29ebd9769926496,0,2,185.707943,176.909439,52.0,271.000000,323.000000,85.292057
2,4cc0c8e2dd8ffc9c,0,3,382.617382,177.120714,102.0,382.617382,484.617382,0.000000
3,28a358995e41a9f5,0,4,579.738096,197.526428,267.0,579.738096,846.738096,0.000000
4,cf2dd7eb2a698665,1,1,0.000000,172.957740,198.0,0.000000,198.000000,0.000000
5,a8da758576964b78,1,2,192.957740,177.001851,368.0,218.000000,586.000000,25.042260
6,564d6a7a6173eed1,1,3,389.959592,227.901180,404.0,606.000000,1010.000000,216.040408
7,f42047cb579b3055,1,4,637.860631,139.360367,95.0,1030.000000,1125.000000,392.139369
8,fc7299c4e0060a00,2,1,0.000000,169.393854,66.0,0.000000,66.000000,0.000000
9,de7a28cae633f05a,2,2,189.393854,219.187313,184.0,189.393854,373.393854,0.000000



ROOM-LEVEL REALISED RESULTS


,Cohort,Model,Lambda,Room,N_Cases,Actual_Surgical_Workload,Realised_Room_Finish,Realised_Overtime,Total_Start_Delay,Mean_Start_Delay,Max_Start_Delay
0,Original_Fixed_Cohort,LightGBM,0.0,0,4,672.0,846.738096,366.738096,85.292057,21.323014,85.292057
1,Original_Fixed_Cohort,LightGBM,0.0,1,4,1065.0,1125.000000,645.000000,633.222037,158.305509,392.139369
2,Original_Fixed_Cohort,LightGBM,0.0,2,4,455.0,718.314564,238.314564,0.000000,0.000000,0.000000


In [10]:
# ============================================================
# 8. SMOKE-TEST INTEGRITY CHECK
# ============================================================

assert len(smoke_schedule) == N_CASES

assert (
    smoke_schedule["LOG_ID"]
    .nunique()
    ==
    N_CASES
)

assert (
    smoke_schedule[
        "Actual_Duration"
    ]
    .sum()
    ==
    actual_fixed[
        "ACTUAL_DURATION"
    ]
    .sum()
)

assert (
    smoke_schedule[
        "Replay_Turnover_Violations"
    ].sum()
    == 0
) if "Replay_Turnover_Violations" in smoke_schedule.columns else True


assert (
    smoke_summary[
        "Replay_Turnover_Violations"
    ]
    ==
    0
)


assert np.isclose(
    smoke_summary[
        "Actual_Total_Duration"
    ],
    actual_fixed[
        "ACTUAL_DURATION"
    ].sum()
)


print("=" * 75)
print("STEP 4C SMOKE TEST: PASS")
print("=" * 75)

print(
    "No realised overlap / turnover violations detected."
)

print(
    "Actual durations were used only after MILP optimisation."
)

print(
    "Schedule capture and replay pipeline is operational."
)

STEP 4C SMOKE TEST: PASS
No realised overlap / turnover violations detected.
Actual durations were used only after MILP optimisation.
Schedule capture and replay pipeline is operational.


In [11]:
# ============================================================
# 8B. PROMOTE PASSED SMOKE TEST TO OFFICIAL PHASE-A OUTPUT
#
# This prevents the validated LightGBM lambda=0 run from
# being unnecessarily solved again in the full experiment.
# ============================================================

assert smoke_summary["Replay_Turnover_Violations"] == 0
assert len(smoke_schedule) == N_CASES
assert smoke_schedule["LOG_ID"].nunique() == N_CASES

smoke_summary["Validation_Run_Success"] = True
smoke_summary["Failure_Type"] = ""
smoke_summary["Failure_Message"] = ""

# ------------------------------------------------------------
# Save case-level schedule
# ------------------------------------------------------------

smoke_schedule.to_csv(
    SCHEDULE_OUTPUT,
    index=False
)

# ------------------------------------------------------------
# Save room-level results
# ------------------------------------------------------------

smoke_rooms.to_csv(
    ROOM_OUTPUT,
    index=False
)

# ------------------------------------------------------------
# Save run-level summary
# ------------------------------------------------------------

pd.DataFrame(
    [smoke_summary]
).to_csv(
    SUMMARY_OUTPUT,
    index=False
)

# ------------------------------------------------------------
# Create checkpoint
# ------------------------------------------------------------

smoke_checkpoint = pd.DataFrame(
    [
        {
            "Model": SMOKE_MODEL,
            "Lambda": SMOKE_LAMBDA,
            "Completed": True,
            "Validation_Run_Success": True,
            "MILP_Gap": smoke_summary["MILP_Gap"],
            "MILP_Solve_Time": smoke_summary["MILP_Solve_Time"],
            "Realised_Makespan": smoke_summary["Realised_Makespan"],
            "Realised_Overtime": smoke_summary["Realised_Overtime"]
        }
    ]
)

smoke_checkpoint.to_csv(
    CHECKPOINT_OUTPUT,
    index=False
)

print("=" * 75)
print("SMOKE TEST PROMOTED TO OFFICIAL PHASE-A RESULT")
print("=" * 75)

print(f"Model              : {SMOKE_MODEL}")
print(f"Lambda             : {SMOKE_LAMBDA}")
print(f"Cases saved        : {len(smoke_schedule)}")
print(f"Rooms saved        : {len(smoke_rooms)}")
print(f"Checkpoint created : {CHECKPOINT_OUTPUT}")

SMOKE TEST PROMOTED TO OFFICIAL PHASE-A RESULT
Model              : LightGBM
Lambda             : 0.0
Cases saved        : 12
Rooms saved        : 3
Checkpoint created : Results\additional_realised_validation_checkpoint.csv


In [19]:
# ============================================================
# 9. FULL PHASE-A ENDPOINT VALIDATION
#
# 3 models × 2 endpoint lambdas = 6 runs
#
# IMPORTANT:
# The LightGBM lambda=0 smoke test will be rerun here unless
# already present in the checkpoint.
#
# Checkpoint/resume logic prevents accidental duplicate work.
# ============================================================


RUN_PLAN = [
    ("LightGBM", 0.0),
    ("LightGBM", 0.5),
    ("LightGBM", 1.0),

    ("RF", 0.0),
    ("RF", 0.5),
    ("RF", 1.0),

    ("XGBoost", 0.0),
    ("XGBoost", 0.5),
    ("XGBoost", 1.0)
]


# ------------------------------------------------------------
# Load existing checkpoint if present
# ------------------------------------------------------------

if os.path.exists(
    CHECKPOINT_OUTPUT
):

    checkpoint_df = pd.read_csv(
        CHECKPOINT_OUTPUT
    )

    print(
        f"Existing checkpoint loaded: "
        f"{len(checkpoint_df)} completed runs."
    )

else:

    checkpoint_df = pd.DataFrame()

    print(
        "No existing checkpoint found."
    )


# ------------------------------------------------------------
# Determine completed runs
# ------------------------------------------------------------

completed_keys = set()


if len(checkpoint_df) > 0:

    required_checkpoint_cols = {
        "Model",
        "Lambda"
    }

    assert required_checkpoint_cols.issubset(
        checkpoint_df.columns
    )


    for _, row in checkpoint_df.iterrows():

        completed_keys.add(
            (
                str(row["Model"]),
                round(
                    float(row["Lambda"]),
                    6
                )
            )
        )


print(
    f"Completed keys: {completed_keys}"
)


# ------------------------------------------------------------
# Existing detailed outputs
# ------------------------------------------------------------

if os.path.exists(SCHEDULE_OUTPUT):

    all_schedule_df = pd.read_csv(
        SCHEDULE_OUTPUT
    )

else:

    all_schedule_df = pd.DataFrame()


if os.path.exists(ROOM_OUTPUT):

    all_room_df = pd.read_csv(
        ROOM_OUTPUT
    )

else:

    all_room_df = pd.DataFrame()


if os.path.exists(SUMMARY_OUTPUT):

    all_summary_df = pd.read_csv(
        SUMMARY_OUTPUT
    )

else:

    all_summary_df = pd.DataFrame()


# ------------------------------------------------------------
# Run loop
# ------------------------------------------------------------

experiment_start = time.time()


for run_number, (
    model_name,
    lam
) in enumerate(
    RUN_PLAN,
    start=1
):

    lam = float(lam)

    run_key = (
        model_name,
        round(lam, 6)
    )


    print("\n" + "=" * 75)

    print(
        f"RUN {run_number}/{len(RUN_PLAN)}"
    )

    print(
        f"Model  = {model_name}"
    )

    print(
        f"Lambda = {lam:.1f}"
    )

    print("=" * 75)


    # --------------------------------------------------------
    # Resume protection
    # --------------------------------------------------------

    if run_key in completed_keys:

        print(
            "SKIPPED — already completed "
            "according to checkpoint."
        )

        continue


    df_day = (
        verified_model_data[
            model_name
        ]
        .copy()
    )


    run_start = time.time()


    try:

        # ====================================================
        # MILP SOLVE
        # ====================================================

        result = solve_or_milp(

            df_day=df_day,

            lam=lam,

            n_rooms=N_ROOMS,

            room_capacity=ROOM_CAPACITY,

            turnover=TURNOVER,

            balance_weight=BALANCE_WEIGHT,

            duration_cap=DURATION_CAP,

            time_limit=TIME_LIMIT
        )


        if result is None:

            raise RuntimeError(
                "MILP returned no feasible incumbent."
            )


        # ====================================================
        # ACTUAL-DURATION REPLAY
        # ====================================================

        (
            schedule_df,
            room_df,
            summary
        ) = replay_realised_schedule(

            df_day=df_day,

            milp_result=result,

            actual_duration_map=actual_duration_map,

            model_name=model_name,

            lam=lam,

            cohort_name="Original_Fixed_Cohort",

            n_rooms=N_ROOMS,

            room_capacity=ROOM_CAPACITY,

            turnover=TURNOVER,

            duration_cap=DURATION_CAP
        )


        summary[
            "Validation_Run_Success"
        ] = True

        summary[
            "Failure_Type"
        ] = ""

        summary[
            "Failure_Message"
        ] = ""


        # ====================================================
        # APPEND DETAILED OUTPUTS
        # ====================================================

        if len(all_schedule_df) == 0:

            all_schedule_df = (
                schedule_df.copy()
            )

        else:

            all_schedule_df = pd.concat(
                [
                    all_schedule_df,
                    schedule_df
                ],
                ignore_index=True
            )


        if len(all_room_df) == 0:

            all_room_df = (
                room_df.copy()
            )

        else:

            all_room_df = pd.concat(
                [
                    all_room_df,
                    room_df
                ],
                ignore_index=True
            )


        summary_row = pd.DataFrame(
            [summary]
        )


        if len(all_summary_df) == 0:

            all_summary_df = (
                summary_row.copy()
            )

        else:

            all_summary_df = pd.concat(
                [
                    all_summary_df,
                    summary_row
                ],
                ignore_index=True
            )


        # ====================================================
        # SAVE IMMEDIATELY
        # ====================================================

        all_schedule_df.to_csv(
            SCHEDULE_OUTPUT,
            index=False
        )

        all_room_df.to_csv(
            ROOM_OUTPUT,
            index=False
        )

        all_summary_df.to_csv(
            SUMMARY_OUTPUT,
            index=False
        )


        # ----------------------------------------------------
        # Update checkpoint only AFTER all detailed outputs
        # have been written successfully.
        # ----------------------------------------------------

        checkpoint_row = pd.DataFrame(
            [
                {
                    "Model":
                        model_name,

                    "Lambda":
                        lam,

                    "Completed":
                        True,

                    "Validation_Run_Success":
                        True,

                    "MILP_Gap":
                        summary[
                            "MILP_Gap"
                        ],

                    "MILP_Solve_Time":
                        summary[
                            "MILP_Solve_Time"
                        ],

                    "Realised_Makespan":
                        summary[
                            "Realised_Makespan"
                        ],

                    "Realised_Overtime":
                        summary[
                            "Realised_Overtime"
                        ]
                }
            ]
        )


        if len(checkpoint_df) == 0:

            checkpoint_df = (
                checkpoint_row.copy()
            )

        else:

            checkpoint_df = pd.concat(
                [
                    checkpoint_df,
                    checkpoint_row
                ],
                ignore_index=True
            )


        checkpoint_df.to_csv(
            CHECKPOINT_OUTPUT,
            index=False
        )


        completed_keys.add(
            run_key
        )


        # ====================================================
        # RUN OUTPUT
        # ====================================================

        elapsed_run = (
            time.time()
            -
            run_start
        )


        print("\nRUN COMPLETE")

        print(
            f"Solve time            : "
            f"{summary['MILP_Solve_Time']:.2f} s"
        )

        print(
            f"Final MIP gap         : "
            f"{summary['MILP_Gap']:.4f}"
        )

        print(
            f"Hit time limit        : "
            f"{summary['MILP_Hit_Time_Limit']}"
        )

        print(
            f"Capped cases          : "
            f"{summary['MILP_N_Capped']}"
        )

        print(
            f"Realised makespan     : "
            f"{summary['Realised_Makespan']:.2f}"
        )

        print(
            f"Realised overtime     : "
            f"{summary['Realised_Overtime']:.2f}"
        )

        print(
            f"Total start delay     : "
            f"{summary['Total_Start_Delay']:.2f}"
        )

        print(
            f"Turnover violations   : "
            f"{summary['Replay_Turnover_Violations']}"
        )

        print(
            f"Total wall-clock run  : "
            f"{elapsed_run:.2f} s"
        )


    except Exception as exc:

        print("\nRUN FAILED")

        print(
            f"{type(exc).__name__}: {exc}"
        )

        raise


print("\n" + "=" * 75)
print("PHASE-A RUN LOOP FINISHED")
print("=" * 75)

print(
    f"Elapsed: "
    f"{(time.time() - experiment_start) / 60:.2f} min"
)

Existing checkpoint loaded: 6 completed runs.
Completed keys: {('XGBoost', 0.0), ('RF', 1.0), ('LightGBM', 1.0), ('LightGBM', 0.0), ('RF', 0.0), ('XGBoost', 1.0)}

RUN 1/9
Model  = LightGBM
Lambda = 0.0
SKIPPED — already completed according to checkpoint.

RUN 2/9
Model  = LightGBM
Lambda = 0.5

RUN COMPLETE
Solve time            : 300.01 s
Final MIP gap         : 0.4781
Hit time limit        : True
Capped cases          : 0
Realised makespan     : 1141.73
Realised overtime     : 1616.56
Total start delay     : 250.41
Turnover violations   : 0
Total wall-clock run  : 301.45 s

RUN 3/9
Model  = LightGBM
Lambda = 1.0
SKIPPED — already completed according to checkpoint.

RUN 4/9
Model  = RF
Lambda = 0.0
SKIPPED — already completed according to checkpoint.

RUN 5/9
Model  = RF
Lambda = 0.5

RUN COMPLETE
Solve time            : 300.02 s
Final MIP gap         : 0.6067
Hit time limit        : True
Capped cases          : 0
Realised makespan     : 1067.16
Realised overtime     : 1182.43
Total 

In [23]:
# ============================================================
# 10. FINAL PHASE-A INTEGRITY AUDIT
# ============================================================

summary_df = pd.read_csv(
    SUMMARY_OUTPUT
)

schedule_df = pd.read_csv(
    SCHEDULE_OUTPUT
)

room_df = pd.read_csv(
    ROOM_OUTPUT
)


# ------------------------------------------------------------
# Remove accidental duplicate Model × Lambda rows if notebook
# was manually rerun after successful completion.
#
# Keep the latest saved result.
# ------------------------------------------------------------

summary_df = (
    summary_df
    .drop_duplicates(
        subset=[
            "Cohort",
            "Model",
            "Lambda"
        ],
        keep="last"
    )
    .reset_index(drop=True)
)


schedule_df = (
    schedule_df
    .drop_duplicates(
        subset=[
            "Cohort",
            "Model",
            "Lambda",
            "LOG_ID"
        ],
        keep="last"
    )
    .reset_index(drop=True)
)


room_df = (
    room_df
    .drop_duplicates(
        subset=[
            "Cohort",
            "Model",
            "Lambda",
            "Room"
        ],
        keep="last"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Expected Phase-A endpoint runs
# ------------------------------------------------------------

expected_keys = set(
    (
        model_name,
        float(lam)
    )
    for model_name, lam
    in RUN_PLAN
)


actual_keys = set(
    zip(
        summary_df["Model"],
        summary_df["Lambda"]
        .astype(float)
    )
)


missing_keys = (
    expected_keys
    -
    actual_keys
)


assert not missing_keys, (
    "Missing Phase-A runs:\n"
    f"{sorted(missing_keys)}"
)


# ------------------------------------------------------------
# Each run must contain exactly 12 cases
# ------------------------------------------------------------

case_counts = (
    schedule_df
    .groupby(
        [
            "Model",
            "Lambda"
        ]
    )
    ["LOG_ID"]
    .nunique()
)


assert (
    case_counts == N_CASES
).all(), (
    "At least one run does not contain exactly "
    f"{N_CASES} unique cases."
)


# ------------------------------------------------------------
# Each run must contain exactly 3 room summaries
# ------------------------------------------------------------

room_counts = (
    room_df
    .groupby(
        [
            "Model",
            "Lambda"
        ]
    )
    ["Room"]
    .nunique()
)


assert (
    room_counts == N_ROOMS
).all(), (
    "At least one run does not contain exactly "
    f"{N_ROOMS} rooms."
)


# ------------------------------------------------------------
# No replay violations
# ------------------------------------------------------------

assert (
    summary_df[
        "Replay_Turnover_Violations"
    ]
    ==
    0
).all()


# ------------------------------------------------------------
# Same actual workload in every run
# ------------------------------------------------------------

actual_totals = (
    summary_df[
        "Actual_Total_Duration"
    ]
    .to_numpy(dtype=float)
)


assert np.allclose(
    actual_totals,
    actual_totals[0]
)


# ------------------------------------------------------------
# Save cleaned outputs
# ------------------------------------------------------------

summary_df.to_csv(
    SUMMARY_OUTPUT,
    index=False
)

schedule_df.to_csv(
    SCHEDULE_OUTPUT,
    index=False
)

room_df.to_csv(
    ROOM_OUTPUT,
    index=False
)


print("=" * 75)
print("PHASE-A FINAL INTEGRITY AUDIT: PASS")
print("=" * 75)

print(
    f"Successful endpoint runs: "
    f"{len(expected_keys)}"
)

print(
    f"Case-level rows: "
    f"{len(schedule_df)}"
)

print(
    f"Room-level rows: "
    f"{len(room_df)}"
)

print(
    "Replay turnover violations: 0"
)

print(
    f"Actual workload per run: "
    f"{actual_totals[0]:.2f} min"
)

PHASE-A FINAL INTEGRITY AUDIT: PASS
Successful endpoint runs: 9
Case-level rows: 108
Room-level rows: 27
Replay turnover violations: 0
Actual workload per run: 2192.00 min


In [24]:
# ============================================================
# 11. ENDPOINT REALISED-VALIDATION COMPARISON
# ============================================================

comparison_columns = [

    "Model",
    "Lambda",

    "MILP_N_Capped",

    "MILP_Planned_Makespan",
    "MILP_Planned_Overtime",

    "Realised_Makespan",
    "Realised_Overtime",

    "Total_Start_Delay",
    "Mean_Start_Delay",
    "Max_Start_Delay",

    "Delayed_Cases",

    "MILP_Gap",
    "MILP_Hit_Time_Limit"
]


comparison_df = (
    summary_df[
        comparison_columns
    ]
    .copy()
)


model_order = {
    "LightGBM": 0,
    "RF": 1,
    "XGBoost": 2
}


comparison_df[
    "_Model_Order"
] = (
    comparison_df[
        "Model"
    ]
    .map(model_order)
)


comparison_df = (
    comparison_df
    .sort_values(
        [
            "_Model_Order",
            "Lambda"
        ]
    )
    .drop(
        columns=[
            "_Model_Order"
        ]
    )
    .reset_index(drop=True)
)


print("=" * 75)
print("ADDITIONAL REALISED VALIDATION — ENDPOINT RESULTS")
print("=" * 75)

display(
    comparison_df
)

ADDITIONAL REALISED VALIDATION — ENDPOINT RESULTS


,Model,Lambda,MILP_N_Capped,MILP_Planned_Makespan,MILP_Planned_Overtime,Realised_Makespan,Realised_Overtime,Total_Start_Delay,Mean_Start_Delay,Max_Start_Delay,Delayed_Cases,MILP_Gap,MILP_Hit_Time_Limit
0,LightGBM,0.0,0,777.264524,891.570972,1125.000000,1250.052660,718.514094,59.876175,392.139369,4,0.600397,True
1,LightGBM,0.5,0,1130.072839,1946.336887,1141.727962,1616.559004,250.413029,20.867752,128.660969,4,0.478095,True
2,LightGBM,1.0,4,1438.870968,2869.495083,1520.976265,2409.833777,81.223119,6.768593,81.223119,1,0.453939,True
3,RF,0.0,0,735.699304,767.065779,1125.000000,1352.722176,733.541207,61.128434,461.636845,4,0.625020,True
4,RF,0.5,0,781.969837,847.459479,1067.158628,1182.430073,557.408732,46.450728,219.329158,6,0.606748,True
5,RF,1.0,0,831.609420,927.889205,1071.399382,1392.072046,318.609424,26.550785,211.549554,4,0.627410,True
6,XGBoost,0.0,0,780.623867,819.522137,1115.504710,1304.886977,406.278658,33.856555,278.369620,4,0.598912,True
7,XGBoost,0.5,0,1125.806089,1935.627437,1115.451530,1817.923189,222.566486,18.547207,121.553055,2,0.549075,True
8,XGBoost,1.0,9,1482.645801,3007.937402,1490.645801,2746.937402,61.354424,5.112869,61.354424,1,0.448362,True


In [25]:
# ============================================================
# 12. COMPACT REPORTING TABLE
# ============================================================

report_table = (
    summary_df[
        [
            "Model",
            "Lambda",
            "Realised_Makespan",
            "Realised_Overtime",
            "Total_Start_Delay",
            "Delayed_Cases",
            "MILP_N_Capped",
            "MILP_Gap"
        ]
    ]
    .copy()
)


report_table[
    "Realised_Makespan"
] = (
    report_table[
        "Realised_Makespan"
    ]
    .round(2)
)


report_table[
    "Realised_Overtime"
] = (
    report_table[
        "Realised_Overtime"
    ]
    .round(2)
)


report_table[
    "Total_Start_Delay"
] = (
    report_table[
        "Total_Start_Delay"
    ]
    .round(2)
)


report_table[
    "MILP_Gap"
] = (
    100
    *
    report_table[
        "MILP_Gap"
    ]
).round(2)


report_table = (
    report_table
    .sort_values(
        [
            "Lambda",
            "Model"
        ]
    )
    .reset_index(drop=True)
)


display(
    report_table
)

,Model,Lambda,Realised_Makespan,Realised_Overtime,Total_Start_Delay,Delayed_Cases,MILP_N_Capped,MILP_Gap
0,LightGBM,0.0,1125.00,1250.05,718.51,4,0,60.04
1,RF,0.0,1125.00,1352.72,733.54,4,0,62.50
2,XGBoost,0.0,1115.50,1304.89,406.28,4,0,59.89
3,LightGBM,0.5,1141.73,1616.56,250.41,4,0,47.81
4,RF,0.5,1067.16,1182.43,557.41,6,0,60.67
5,XGBoost,0.5,1115.45,1817.92,222.57,2,0,54.91
6,LightGBM,1.0,1520.98,2409.83,81.22,1,4,45.39
7,RF,1.0,1071.40,1392.07,318.61,4,0,62.74
8,XGBoost,1.0,1490.65,2746.94,61.35,1,9,44.84
